##### **HOMEWORK 1**

**Mateo Vivanco (00328476)** <br>
**Roberto Villafuerte (00329429)**

## Section 1: Research - Historical & Advanced Classical Ciphers

### The Vigenère Cipher:

**Mechanism:**

The Tabula Recta (Vigenère Square) is a 26×26 grid where each of the 26 rows is the standard alphabet shifted left by one additional position — row 0 is the normal alphabet (A–Z), row 1 shifts by 1 (B–Z, A), and so on. To encrypt, the keyword is repeated to match the length of the plaintext. For each plaintext letter, the corresponding keyword letter determines which row of the table to use; the plaintext letter then selects the column. The letter at that row–column intersection is the ciphertext.

**Example:** Keyword `KEY`, plaintext `HELLO`
- H + K (row 10) → R
- E + E (row 4) → I
- L + Y (row 24) → J
- L + K (row 10) → V
- O + E (row 4) → S  → Ciphertext: `RIJVS`

**Analysis (defense against frequency analysis):**
In a simple Caesar or monoalphabetic substitution cipher, each plaintext letter always maps to the same ciphertext letter, so the natural frequency distribution of a language (e.g., 'E' is most common in English) is preserved in the ciphertext and can be exploited. The Vigenère cipher is *polyalphabetic*: the same plaintext letter is encrypted differently depending on its position relative to the repeating keyword. A plaintext 'E' might become 'J' in one position and 'R' in another. This "flattens" the frequency distribution across the ciphertext, making simple frequency analysis ineffective.

**Breaking the Cipher — Kasiski Examination:**
1. **Find repeated sequences:** Scan the ciphertext for repeated trigrams or longer repeated substrings. Repetitions occur when identical sections of plaintext align with the same section of the repeated keyword.
2. **Record distances:** Note the distance (in characters) between each pair of repeated sequences.
3. **Compute GCD:** The keyword length is likely a divisor of each of those distances. Taking the GCD (or most common factor) of all the recorded distances gives a strong candidate for the keyword length $n$.
4. **Divide and conquer:** Once $n$ is known, the ciphertext is split into $n$ independent groups — each group is effectively a Caesar cipher and can be broken by standard frequency analysis.

### The Hill Cipher:

**Mathematical Basis:**
The Hill cipher operates on blocks of $n$ plaintext letters at a time.

1. **Vector conversion:** Each letter is assigned a numerical value (A=0, B=1, …, Z=25). A block of $n$ plaintext letters is arranged as a column vector $\mathbf{p} \in \mathbb{Z}_{26}^n$.
2. **Encryption:** The sender and receiver share an $n \times n$ key matrix $K$ with entries in $\mathbb{Z}_{26}$. The ciphertext block is computed as:
$$\mathbf{c} = K \cdot \mathbf{p} \pmod{26}$$
3. **Decryption:** The recipient recovers the plaintext via $\mathbf{p} = K^{-1} \cdot \mathbf{c} \pmod{26}$.

**Example (2×2 block):** Plaintext `HI` → vector $\mathbf{p} = \begin{pmatrix}7\\8\end{pmatrix}$. With key $K = \begin{pmatrix}3&3\\2&5\end{pmatrix}$: $\mathbf{c} = K\mathbf{p} \pmod{26} = \begin{pmatrix}45\\54\end{pmatrix} \pmod{26} = \begin{pmatrix}19\\2\end{pmatrix}$ → ciphertext `TC`.

**Requirements for the key matrix:**
The matrix $K$ must be **invertible modulo 26**, which requires two conditions:
1. $\det(K) \neq 0$ (non-zero determinant, so the matrix is algebraically invertible).
2. $\gcd(\det(K),\ 26) = 1$ — the determinant must be coprime to 26 (i.e., not divisible by 2 or 13), so that its modular inverse $(\det K)^{-1} \pmod{26}$ exists.

If either condition fails, $K^{-1} \pmod{26}$ does not exist and decryption is impossible.


### The Playfair Cipher:

**Grid construction:**
1. Write out the keyword, removing any duplicate letters.
2. Append the remaining letters of the alphabet (in order) that do not appear in the keyword. **I and J share a single cell** (the alphabet is reduced to 25 letters).
3. Fill the letters row-by-row into a $5 \times 5$ grid.

**Encryption rules (digraph substitution):**
First, split the plaintext into pairs of letters. If both letters in a pair are the same, insert a filler (usually **X**) between them and re-pair. If the message length is odd, append X at the end. Then apply:

| Condition | Rule |
|-----------|------|
| Same row | Replace each letter with the letter **immediately to its right** in the same row (wrap around). |
| Same column | Replace each letter with the letter **immediately below** it in the same column (wrap around). |
| Rectangle | Each letter is replaced by the letter in its **own row** but in the **other letter's column**. |

Decryption uses the inverse (left/up instead of right/down for the first two cases; the rectangle rule is self-inverse).

**Historical context — WWI and WWII:**
The Playfair cipher was invented by Charles Wheatstone in 1854 and championed by Lord Playfair. It was adopted by the British Army and saw extensive use from the Boer War through both World Wars. It was considered "field-ready" for several reasons:
- **No equipment required:** Operators needed only a pencil, paper, and a memorized keyword — no mechanical device or cipher machine.
- **Speed:** Trained soldiers could encrypt and decrypt messages quickly under field conditions.
- **Sufficient security:** It defeated casual interception and simple frequency analysis (since it operates on digraphs, not individual letters, it produces ~600 possible pairs versus 26 single letters, significantly complicating analysis).
- **Simplicity of training:** The five rules are easy to memorize and apply, unlike more complex book ciphers or machine systems that required specialist training.

### The Enigma Machine:

**Rotor Logic:**
Each rotor is a physical disc wired to perform a fixed letter-substitution (a permutation of the 26-letter alphabet). The Enigma typically used three (later four) rotors in series. When a key is pressed, an electrical signal passes through the rotors from right to left, through the reflector, and back left to right; the cumulative substitutions of all rotors determine the output (lit) letter.

Critically, **the rightmost rotor advances one step with every keystroke** — like the ones digit of an odometer. When it completes a full revolution (or reaches its notch position), it causes the middle rotor to step, and similarly for the left rotor. Because the rotor positions change, the effective substitution alphabet is **different for every single character encrypted**, producing a *polyalphabetic* cipher with an astronomically long period (e.g., for three rotors: $26^3 = 17{,}576$ steps before the pattern repeats, and with rotor selection the period is far longer).

**The Reflector:**
The reflector is a fixed (non-rotating) disc at the left end of the rotor stack that pairs up all 26 letters into 13 pairs and routes the signal back through the rotors in the reverse direction. Two consequences follow:
1. **Symmetric encryption/decryption:** Because the signal makes a return trip through the same rotors, the same machine settings encrypt and decrypt — sending operators did not need two different configurations.
2. **A letter can never encrypt as itself:** The reflector's wiring ensures that no letter is paired with itself. Combined with the symmetric return path, this means the ciphertext letter is always *different* from the plaintext letter. While convenient for operators, this was a fatal cryptographic weakness: Allied codebreakers at Bletchley Park exploited it to eliminate candidate settings (if a ciphertext letter matched the plaintext letter, that setting was impossible).

**The Plugboard (Steckerbrett):**
Before the signal entered the rotors (and again on the way out), it passed through the plugboard, where pairs of letters could be swapped using physical patch cables. In standard WWII operation, **10 cables** were used, swapping 10 pairs of letters (leaving 6 letters unchanged).

The number of ways to choose and pair 10 letters from 26 is:
$$\frac{26!}{(26-20)! \cdot 2^{10} \cdot 10!} = 150{,}738{,}274{,}937{,}250 \approx 1.5 \times 10^{14}$$

This factor of ~150 trillion was multiplied on top of the rotor settings, making the total keyspace approximately $10^{23}$ — far beyond any brute-force attack feasible at the time. The plugboard was therefore the single largest contributor to the Enigma's overall security.

## Section 2: Mathematical Foundations

### Modular Arithmetic

**a) $x \equiv 457 \pmod{13}$**

Divide 457 by 13:
$$457 = 35 \times 13 + 2 \quad \text{(since } 35 \times 13 = 455 \text{)}$$

$$\boxed{x \equiv 2 \pmod{13}}$$

**b) $x \equiv -25 \pmod{7}$**

Find the remainder of 25 divided by 7, then negate mod 7:
$$25 = 3 \times 7 + 4 \implies -25 \equiv -4 \equiv 7 - 4 \pmod{7}$$

Equivalently: $-25 = (-4) \times 7 + 3$

$$\boxed{x \equiv 3 \pmod{7}}$$

### The Euclidean Algorithm

**a) $\gcd(1160, 480)$**

Apply the algorithm repeatedly (divide, take remainder):

| Step | Equation |
|------|----------|
| 1 | $1160 = 2 \times 480 + 200$ |
| 2 | $480 = 2 \times 200 + 80$ |
| 3 | $200 = 2 \times 80 + 40$ |
| 4 | $80 = 2 \times 40 + 0$ |

The remainder is 0, so:

$$\boxed{\gcd(1160,\ 480) = 40}$$

### Extended Euclidean Algorithm — find $x, y$ such that $1160x + 480y = 40$

Work backwards through the steps above, expressing each remainder as a linear combination of the original numbers:

**Step 1 — express 40 from step 3:**
$$40 = 200 - 2 \times 80$$

**Step 2 — substitute 80 from step 2 ($80 = 480 - 2 \times 200$):**
$$40 = 200 - 2 \times (480 - 2 \times 200) = 200 - 2 \times 480 + 4 \times 200 = 5 \times 200 - 2 \times 480$$

**Step 3 — substitute 200 from step 1 ($200 = 1160 - 2 \times 480$):**
$$40 = 5 \times (1160 - 2 \times 480) - 2 \times 480 = 5 \times 1160 - 10 \times 480 - 2 \times 480$$

$$40 = 5 \times 1160 + (-12) \times 480$$

$$\boxed{x = 5, \quad y = -12}$$

**Verification:** $1160 \times 5 + 480 \times (-12) = 5800 - 5760 = 40$ 

### Euler's Totient Function

**Recall:** For $n = p_1^{a_1} p_2^{a_2} \cdots p_k^{a_k}$:
$$\phi(n) = n \prod_{p \mid n} \left(1 - \frac{1}{p}\right)$$

**a) $\phi(21)$**

$$21 = 3 \times 7 \qquad \text{(both prime)}$$

$$\phi(21) = (3-1)(7-1) = 2 \times 6 = \boxed{12}$$

**b) $\phi(17)$**

17 is prime, so:
$$\phi(17) = 17 - 1 = \boxed{16}$$

**c) $\phi(360)$**

Prime factorization:
$$360 = 2^3 \times 3^2 \times 5^1$$

Apply the formula:
$$\phi(360) = 360 \times \left(1 - \frac{1}{2}\right)\left(1 - \frac{1}{3}\right)\left(1 - \frac{1}{5}\right)$$

$$= 360 \times \frac{1}{2} \times \frac{2}{3} \times \frac{4}{5} = 360 \times \frac{8}{30} = \boxed{96}$$


### Shannon Entropy

The Shannon entropy of a discrete random variable $X$ is:
$$H(X) = -\sum_{i} P(x_i) \log_2 P(x_i)$$

| Symbol | $P(x_i)$ | $\log_2 P(x_i)$ | $-P(x_i)\log_2 P(x_i)$ |
|--------|-----------|-----------------|------------------------|
| A | $\frac{1}{2}$ | $-1$ | $\frac{1}{2}$ |
| B | $\frac{1}{4}$ | $-2$ | $\frac{1}{2}$ |
| C | $\frac{1}{8}$ | $-3$ | $\frac{3}{8}$ |
| D | $\frac{1}{8}$ | $-3$ | $\frac{3}{8}$ |

$$H(X) = \frac{1}{2} + \frac{1}{2} + \frac{3}{8} + \frac{3}{8} = 1 + \frac{6}{8} = \frac{7}{4}$$

$$\boxed{H(X) = 1.75 \text{ bits}}$$

Note: This result is optimal — it matches the bit lengths of an ideal Huffman code (A→`0`, B→`10`, C→`110`, D→`111`), confirming the source is a perfect binary tree distribution.


### Bayes' Theorem

**Given:**

| Symbol | Meaning | Value |
|--------|---------|-------|
| $P(A)$ | Probability of an actual attack | $0.001$ |
| $P(\bar{A})$ | Probability of no attack | $0.999$ |
| $P(F \mid A)$ | True Positive rate | $0.99$ |
| $P(F \mid \bar{A})$ | False Positive rate | $0.01$ |

**Find:** $P(A \mid F)$ — probability of a real attack given that the IDS flagged it.

**Step 1 — Total probability of a flag $P(F)$:**

$$P(F) = P(F \mid A)\,P(A) + P(F \mid \bar{A})\,P(\bar{A})$$
$$= (0.99)(0.001) + (0.01)(0.999) = 0.00099 + 0.00999 = 0.01098$$

**Step 2 — Apply Bayes' Theorem:**

$$P(A \mid F) = \frac{P(F \mid A)\,P(A)}{P(F)} = \frac{0.99 \times 0.001}{0.01098} = \frac{0.00099}{0.01098}$$

$$\boxed{P(A \mid F) \approx 0.0902 \approx 9.02\%}$$

**Interpretation:** Even though the IDS is quite accurate (99% true positive, 1% false positive), the base rate of attacks is very low ($P(A) = 0.001$). The vast majority of flagged events are false positives. This is a classic illustration of the **base rate fallacy** — when a rare event is tested with an imperfect detector, most positive results are false. In practice, a second verification step or additional context would be needed before responding to any single IDS alert.


## Section 3: PicoCTF

### Guess My Cheese

**Smart Approach:**

By what the scientists tells us, we know that we are working with a hashing instead of a cipher, and we have a list of cheese. By what the hints tells us, we can use the SHA256 hash function, and we need to append 2 hexadecimal values (0 - 255) to each cheese of the list, that we can then treat as a Rainbow Table.

In [ ]:
import hashlib

cipher_text = '1c908431d1b16ae278d958f2c49fc80a79f4c2d6699040d6655218a4cc77f301'

cases = ['original', 'lowercase', 'uppercase']

cheeses = []

with open('cheese_list.txt', 'r') as f:
    cheeses = [line.strip() for line in f]
    print(f'Loaded {len(cheeses)} cheeses from the list.')

found = False
for cheese in cheeses:
    for salt in range(256):
        for case in cases:
            if case == 'lowercase':
                variant = cheese.lower()
            elif case == 'uppercase':
                variant = cheese.upper()
            else:
                variant = cheese
        
            salted_cheese = variant.encode('utf-8') + bytes([salt])
        
            hashed_cheese = hashlib.sha256(salted_cheese).hexdigest()
        
            if hashed_cheese == cipher_text:
                print(f'Found the cheese: {variant} with salt: {format(salt, "02x")} and case: {case}')
                found = True
                break
        if found:
            break
    if found:
        break

Flag: **picoCTF{cHeEsY328d8e5d}**


**Bruteforce Approach:**

Supposing we have no idea about any of the hints of the puzzle, since we are dealing with cheese, we can look for a list of cheeses online, and we can try to hash them with SHA256 (that we can determine by the length of the hashed cheese), and we can try to salt them in any way possible to try to find the secret cheese.

In [ ]:
import hashlib
import sys

target_hash = "1c908431d1b16ae278d958f2c49fc80a79f4c2d6699040d6655218a4cc77f301"

# Define the encodings and case variants we want to try.
encodings = ["utf-8", "utf-16-le", "utf-16-be", "latin-1"]
case_variants = {
    "original": lambda s: s,
    "lower": lambda s: s.lower(),
    "upper": lambda s: s.upper(),
}

# Read cheese names from cheese_list.txt (one per line)
try:
    with open("cheese_list.txt", "r") as f:
        cheeses = [line.strip() for line in f if line.strip()]
except FileNotFoundError:
    print("cheese_list.txt not found. Please create it with your list of cheese names.")
    sys.exit(1)

def test_candidate(candidate_bytes, method, extra, cheese, case_name, enc, salt):
    """Tests a candidate bytes object against the target hash."""
    hash_val = hashlib.sha256(candidate_bytes).hexdigest()
    if hash_val == target_hash:
        print("Match found!")
        print(f"Cheese: {cheese}  (case variant: {case_name}, encoding: {enc})")
        print(f"Salt value: {salt} (0x{salt:02x})")
        print(f"Method: {method}, extra: {extra}")
        try:
            candidate_str = candidate_bytes.decode(enc)
        except Exception:
            candidate_str = repr(candidate_bytes)
        print(f"Candidate (using {enc}): {candidate_str}")
        return True
    return False

def brute_force_cheese():
    for cheese in cheeses:
        for case_name, case_func in case_variants.items():
            cheese_variant = case_func(cheese)
            for enc in encodings:
                try:
                    cheese_bytes = cheese_variant.encode(enc)
                except Exception:
                    continue
                for salt in range(256):
                    salt_raw = bytes([salt])
                    salt_hex_str = format(salt, "02x")
                    try:
                        salt_hex = salt_hex_str.encode(enc)
                    except Exception:
                        salt_hex = salt_hex_str.encode("utf-8")

                    # Option A: Append and prepend raw byte.
                    if test_candidate(cheese_bytes + salt_raw, "append_raw", "raw byte appended",
                                      cheese, case_name, enc, salt):
                        return
                    if test_candidate(salt_raw + cheese_bytes, "prepend_raw", "raw byte prepended",
                                      cheese, case_name, enc, salt):
                        return

                    # Option B: Append and prepend hex string.
                    if test_candidate(cheese_bytes + salt_hex, "append_hex", "hex string appended",
                                      cheese, case_name, enc, salt):
                        return
                    if test_candidate(salt_hex + cheese_bytes, "prepend_hex", "hex string prepended",
                                      cheese, case_name, enc, salt):
                        return

                    # Option C: Insert raw byte at every possible index.
                    for i in range(len(cheese_bytes) + 1):
                        candidate = cheese_bytes[:i] + salt_raw + cheese_bytes[i:]
                        if test_candidate(candidate, "insert_raw", f"at byte index {i}",
                                          cheese, case_name, enc, salt):
                            return

                    # Option D: Insert hex string at every possible index.
                    for i in range(len(cheese_bytes) + 1):
                        candidate = cheese_bytes[:i] + salt_hex + cheese_bytes[i:]
                        if test_candidate(candidate, "insert_hex", f"at byte index {i}",
                                          cheese, case_name, enc, salt):
                            return

    print("No matching cheese and salt combination was found.")

brute_force_cheese()

Flag: **picoCTF{cHeEsY328d8e5d}**


### HideToSee

**Atbash Approach:**

Using a tool called "steghide" we can analyse the LSB of a .jpg file and extract hidden text. There we had found the encrypted flag, that we can decrypt using the same encryption method as the image itself suggests: Atbash Cipher.

In [22]:
import subprocess
import os

# Step 1: Extract hidden text from the image using steghide
# steghide is a Linux tool — install with: apt install steghide
# Resolve paths relative to this notebook's location
notebook_dir = os.path.dirname(os.path.abspath('HW1.ipynb'))
image_path = os.path.join(notebook_dir, 'atbash.jpg')
output_file = os.path.join(notebook_dir, 'extracted_flag.txt')

try:
    result = subprocess.run(
        ['steghide', 'extract', '-sf', image_path, '-p', '', '-xf', output_file, '-f'],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(output_file):
        with open(output_file, 'r') as f:
            cipher_text = f.read().strip()
        print(f'Steghide extracted cipher text: {cipher_text}')
    else:
        raise RuntimeError(result.stderr.strip())
except (FileNotFoundError, RuntimeError) as e:
    # Fallback: cipher text previously extracted via steghide on Linux
    cipher_text = 'krxlXGU{zgyzhs_xizxp_05y2z65z}'
    print(f'steghide unavailable or failed ({e}), using previously extracted cipher text: {cipher_text}')

# Step 2: Atbash cipher decryption (A↔Z, B↔Y, C↔X, ...)
alphabet = 'abcdefghijklmnopqrstuvwxyz'
plain_text = ''

for char in cipher_text:
    if char.lower() in alphabet:
        index = alphabet.index(char.lower())
        plain_text += alphabet[25 - index] if char.islower() else alphabet[25 - index].upper()
    else:
        plain_text += char

print(f'Decrypted flag: {plain_text}')

steghide unavailable or failed ([Errno 2] No such file or directory: 'steghide'), using previously extracted cipher text: krxlXGU{zgyzhs_xizxp_05y2z65z}
Decrypted flag: picoCTF{atbash_crack_05b2a65a}


**Bruteforce Approach:**

Again, using the same tool "steghide", we can find the cipher text. Then we can try to bruteforce the substitution cipher by assuming is an Affine Cipher for which we do not know the key, and by knowing that the flag has to start with "picoCTF".

In [19]:
def find_mod_inverse(a, m):
    """Finds the modular multiplicative inverse of a under modulo m."""
    for i in range(1, m):
        if (a * i) % m == 1:
            return i
    return None

def decrypt_affine(ciphertext, a, b):
    """Decrypts text using the Affine formula: D(x) = a^-1 * (x - b) mod 26"""
    a_inv = find_mod_inverse(a, 26)
    if a_inv is None:
        return None

    plaintext = ""
    for char in ciphertext:
        if char.isalpha():
            # Convert to 0-25 format
            base = ord('A') if char.isupper() else ord('a')
            y = ord(char) - base
            
            # Apply the decryption formula
            x = (a_inv * (y - b)) % 26
            
            plaintext += chr(x + base)
        else:
            # Leave symbols (like CTF brackets) alone
            plaintext += char
            
    return plaintext

def brute_force_affine(ciphertext, known_flag_format="picoCTF"):
    print(f"[*] Brute-forcing Affine Cipher for: {ciphertext}\n")
    
    # The 12 valid coprime multipliers for modulo 26
    valid_a_values = [1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
    
    for a in valid_a_values:
        for b in range(26):
            decrypted = decrypt_affine(ciphertext, a, b)
            
            # If we know what we are looking for, catch it automatically
            if known_flag_format in decrypted:
                print("-" * 50)
                print(f"[+] SUCCESS! Found matching key.")
                print(f"[+] Key: a = {a}, b = {b}")
                print(f"[+] Plaintext: {decrypted}")
                print("-" * 50)
                return decrypted
    print("[-] Brute force complete. Known flag format not found.")


cipher_text = "krxlXGU{zgyzhs_xizxp_05y2z65z}"
brute_force_affine(cipher_text)

[*] Brute-forcing Affine Cipher for: krxlXGU{zgyzhs_xizxp_05y2z65z}

--------------------------------------------------
[+] SUCCESS! Found matching key.
[+] Key: a = 25, b = 25
[+] Plaintext: picoCTF{atbash_crack_05b2a65a}
--------------------------------------------------


'picoCTF{atbash_crack_05b2a65a}'

### Rotation

**Brute Force Approach:**

We try all possible value rotations until we find one that starts with the string 'picoCTF'

In [ ]:
cipher_text = 'xqkwKBN{z0bib1wv_l3kzgxb3l_7l140864}'

for i in range(26):
    decrypted = ''.join(
        chr((ord(c) - ord('a') - i) % 26 + ord('a')) if 'a' <= c <= 'z' else chr((ord(c) - ord('A') - i) % 26 + ord('A')) if 'A' <= c <= 'Z' else c
        for c in cipher_text
    )
    print(f"Shift {i}: {decrypted}")

    if "picoctf" in decrypted.lower():
        print(f"Possible flag found with shift {i}: {decrypted}")

**Extracting Rotation Value:**

Since we know that the flag has to start with 'picoCTF', we can extract the offset of the letters to finally find the whole flag.

In [23]:
cipher_text = 'xqkwKBN{z0bib1wv_l3kzgxb3l_7l140864}'

offset = ord('x') - ord('p')  # Calculate the offset based on the first character

decrypted_text = ''
for char in cipher_text:
    if 'a' <= char <= 'z':
        decrypted_char = chr((ord(char) - ord('a') - offset) % 26 + ord('a'))
    elif 'A' <= char <= 'Z':
        decrypted_char = chr((ord(char) - ord('A') - offset) % 26 + ord('A'))
    else:
        decrypted_char = char
    decrypted_text += decrypted_char

print(decrypted_text)

picoCTF{r0tat1on_d3crypt3d_7d140864}


### Substitution2

**English Pattern Recognition:**

We can manually assign some substitution values based on the fact that the flag has to start with 'picoCTF', so that:
- v -> p
- q -> i
- k -> c
- m -> o
- f -> t
- l -> f

And from that, we can guess some letters like n -> h and j -> e, to form the word 'THE' at the beginning. And so on guessing letters on pattern recognition of words in English.

In [ ]:
import re

cipher_text = 'fnjdjjzqsfsjpjdxwmfnjdcjwwjsfxhwqsnjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgsqgkwayqgekthjdvxfdqmfxgyaskthjdknxwwjgejfnjsjkmuvjfqfqmgslmkasvdquxdqwtmgstsfjusxyuqgqsfdxfqmglagyxujgfxwscnqknxdjpjdtasjlawxgyuxdojfxhwjsoqwwsnmcjpjdcjhjwqjpjfnjvdmvjdvadvmsjmlxnqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgqsgmfmgwtfmfjxknpxwaxhwjsoqwwshafxwsmfmejfsfayjgfsqgfjdjsfjyqgxgyjzkqfjyxhmafkmuvafjdskqjgkjyjljgsqpjkmuvjfqfqmgsxdjmlfjgwxhmdqmasxllxqdsxgykmujymcgfmdaggqgeknjkowqsfsxgyjzjkafqgekmglqeskdqvfsmlljgsjmgfnjmfnjdnxgyqsnjxpqwtlmkasjymgjzvwmdxfqmgxgyquvdmpqsxfqmgxgymlfjgnxsjwjujgfsmlvwxtcjhjwqjpjxkmuvjfqfqmgfmaknqgemgfnjmlljgsqpjjwjujgfsmlkmuvafjdsjkadqftqsfnjdjlmdjxhjffjdpjnqkwjlmdfjknjpxgejwqsufmsfayjgfsqgxujdqkxgnqensknmmwsladfnjdcjhjwqjpjfnxfxgagyjdsfxgyqgemlmlljgsqpjfjkngqiajsqsjssjgfqxwlmdumagfqgexgjlljkfqpjyjljgsjxgyfnxffnjfmmwsxgykmglqeadxfqmglmkasjgkmagfjdjyqgyjljgsqpjkmuvjfqfqmgsymjsgmfwjxysfayjgfsfmogmcfnjqdjgjutxsjlljkfqpjwtxsfjxknqgefnjufmxkfqpjwtfnqgowqojxgxffxkojdvqkmkflqsxgmlljgsqpjwtmdqjgfjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgfnxfsjjosfmejgjdxfjqgfjdjsfqgkmuvafjdskqjgkjxumgenqensknmmwjdsfjxknqgefnjujgmaenxhmafkmuvafjdsjkadqftfmvqiajfnjqdkadqmsqftumfqpxfqgefnjufmjzvwmdjmgfnjqdmcgxgyjgxhwqgefnjufmhjffjdyjljgyfnjqduxknqgjsfnjlwxeqsvqkmKFL{G6D4U_4G41T515_15_73Y10A5_42JX1770}'

key = {
    'v': 'p',
    'q': 'i',
    'k': 'c',
    'm': 'o',
    'f': 't',
    'l': 'f',  

    # From now on, we will guess the characters based on a pattern analysis of the text (assuming it's English)
    'j': 'e',
    'n': 'h',
    'u': 'm',
    'g': 'n',
    's': 's',
    'a': 'u',
    'z': 'x',
    'd': 'r',
    't': 'y',
    'e': 'g',
    'w': 'l',
    'x': 'a',
    'h': 'b',
    'y': 'd',
    'p': 'v',
    'o': 'k',
    'c': 'w',
    'i': 'q',
}

plain_text = ''

for char in cipher_text:
    # Check if the character is in our key (handling lowercase for the main text)
    if char.lower() in key:
        plain_text += key[char.lower()].upper()
    else:
        # If we don't know it, leave it as the original character
        plain_text += char

print(plain_text)

print("\n" + "="*60)

# We print the flag found at the end of the text
flag_match = re.search(r'PICOCTF\{.*?\}', plain_text)
if flag_match:
    print(f"[+] Found Flag: {flag_match.group(0)}")

**Frequency Analysis with MCMC:**

We start by assuming the text is written in English and that we are dealing with a substitution cipher. Then we train a Markov Chain Monte Carlo on the book: **The Adventures of Sherlock Holmes** (picked at random) and we generate a frequency model of English for different n-grams (specially bigrams), and then we use that training data to try and decrypt the text with a sort of random evolution process.

In [ ]:
import os
import urllib.request
import re
import math
import json
from collections import Counter

def build_bigram_model(url, output_filename="english_bigrams.json"):
    if os.path.exists(output_filename):
        print(f"[*] '{output_filename}' already exists, skipping download.")
        with open(output_filename, 'r') as f:
            return json.load(f)

    print(f"[*] Downloading training data from: {url}")
    
    try:
        # 1. Fetch the book from Project Gutenberg
        # We use a standard User-Agent so the server doesn't block the automated request
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as response:
            raw_text = response.read().decode('utf-8')
            
        print("[*] Download complete. Cleaning text...")
        
        # 2. Clean the text
        # Strip out all numbers, spaces, and punctuation. Convert to UPPERCASE.
        clean_text = re.sub(r'[^A-Z]', '', raw_text.upper())
        
        # 3. Extract and count Bigrams
        print("[*] Counting n-grams...")
        bigrams = [clean_text[i:i+2] for i in range(len(clean_text)-1)]
        bigram_counts = Counter(bigrams)
        
        total_bigrams = sum(bigram_counts.values())
        print(f"[*] Processed {total_bigrams:,} total bigrams.")
        
        # 4. Calculate Log-Probabilities
        print("[*] Calculating log-probabilities...")
        bigram_log_probs = {}
        
        for bigram, count in bigram_counts.items():
            # Probability = (Times this bigram appears) / (Total bigrams)
            probability = count / total_bigrams
            
            # Convert to log base 10 to prevent underflow errors in MCMC math
            bigram_log_probs[bigram] = round(math.log10(probability), 4)
            
        # 5. Save the model to a flat JSON dictionary
        with open(output_filename, 'w') as json_file:
            json.dump(bigram_log_probs, json_file, indent=4)
            
        print(f"[+] Success! Bigram model saved to '{output_filename}'")
        
        # Show a sneak peek of the top 5 most common bigrams
        print("\n--- Top 5 Bigrams in Training Data ---")
        top_5 = bigram_counts.most_common(5)
        for bg, count in top_5:
            print(f"  '{bg}': {count:,} occurrences (Log-Prob: {bigram_log_probs[bg]})")

    except Exception as e:
        print(f"[!] Error generating model: {e}")

# URL for "The Adventures of Sherlock Holmes" by Arthur Conan Doyle (.txt format)
BOOK_URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"

# Run the generator
build_bigram_model(BOOK_URL)

In [ ]:
import json
import random
import math
import re

def load_ngram_model(filename="english_bigrams.json"):
    """Loads the pre-computed log-probabilities."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[!] Error: Could not find '{filename}'. Did you run the generator script?")
        return None

def score_text(text, bigram_model, default_penalty=-10.0):
    """Calculates fitness using standard dictionary lookups (no matrices)."""
    score = 0.0
    for i in range(len(text) - 1):
        bigram = text[i:i+2]
        # If the bigram exists in our model, add its log-prob. 
        # If it's a bizarre sequence (like 'QZ'), apply a heavy penalty.
        score += bigram_model.get(bigram, default_penalty)
    return score

def mutate_key(current_key):
    """Swaps exactly two random letters in the decryption key."""
    new_key = current_key.copy()
    keys = list(new_key.keys())
    
    a, b = random.sample(keys, 2)
    new_key[a], new_key[b] = new_key[b], new_key[a]
    
    return new_key

def mcmc_decrypt(ciphertext, bigram_model, iterations=15000):
    print(f"[*] Initializing MCMC Cracker with {iterations} iterations...\n")
    
    # Clean text to uppercase letters only
    clean_cipher = ''.join(c for c in ciphertext.upper() if c.isalpha())
    alphabet = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    
    # Step 1: Start with a random key
    shuffled = alphabet.copy()
    random.shuffle(shuffled)
    current_key = {alphabet[i]: shuffled[i] for i in range(26)}
    
    # Apply initial key and score it
    current_decrypted = ''.join(current_key.get(c, c) for c in clean_cipher)
    current_score = score_text(current_decrypted, bigram_model)
    
    best_key = current_key
    best_score = current_score
    
    # Step 2: The MCMC Loop
    for i in range(iterations):
        # Mutate and test
        proposed_key = mutate_key(current_key)
        proposed_decrypted = ''.join(proposed_key.get(c, c) for c in clean_cipher)
        proposed_score = score_text(proposed_decrypted, bigram_model)
        
        # Step 3: Metropolis-Hastings Acceptance
        diff = proposed_score - current_score
        
        # Accept if better, or accept occasionally if worse to escape local peaks
        if diff > 0 or random.uniform(0, 1) < math.exp(diff):
            current_key = proposed_key
            current_score = proposed_score
            
            # Track the absolute best score globally
            if current_score > best_score:
                best_score = current_score
                best_key = current_key.copy()
                
        # Status update every 3000 iterations
        if i % 3000 == 0:
            preview = ''.join(best_key.get(c, c) for c in clean_cipher)[:60]
            print(f"Iteration {i:<6} | Score: {best_score:<8.2f} | Preview: {preview}...")

    print("\n[*] MCMC Evolution Complete.")
    
    # Step 4: Final Decryption Pass
    print("\n" + "="*60)
    print("FINAL DECRYPTED TEXT:")
    final_text = ''.join(best_key.get(c.upper(), c) for c in ciphertext)
    print(final_text)
    print("="*60 + "\n")

    # We print the flag found at the end of the text
    flag_match = re.search(r'PICOCTF\{.*?\}', final_text)
    if flag_match:
        print(f"[+] Found Flag: {flag_match.group(0)}")
    
    return best_key

# --- The Execution Workflow ---

# 1. Load the model we built from Sherlock Holmes
my_model = load_ngram_model("english_bigrams.json")

# 2. Provide the target ciphertext (The picoCTF text from earlier)
target_cipher = "fnjdjjzqsfsjpjdxwmfnjdcjwwjsfxhwqsnjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgsqgkwayqgekthjdvxfdqmfxgyaskthjdknxwwjgejfnjsjkmuvjfqfqmgslmkasvdquxdqwtmgstsfjusxyuqgqsfdxfqmglagyxujgfxwscnqknxdjpjdtasjlawxgyuxdojfxhwjsoqwwsnmcjpjdcjhjwqjpjfnjvdmvjdvadvmsjmlxnqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgqsgmfmgwtfmfjxknpxwaxhwjsoqwwshafxwsmfmejfsfayjgfsqgfjdjsfjyqgxgyjzkqfjyxhmafkmuvafjdskqjgkjyjljgsqpjkmuvjfqfqmgsxdjmlfjgwxhmdqmasxllxqdsxgykmujymcgfmdaggqgeknjkowqsfsxgyjzjkafqgekmglqeskdqvfsmlljgsjmgfnjmfnjdnxgyqsnjxpqwtlmkasjymgjzvwmdxfqmgxgyquvdmpqsxfqmgxgymlfjgnxsjwjujgfsmlvwxtcjhjwqjpjxkmuvjfqfqmgfmaknqgemgfnjmlljgsqpjjwjujgfsmlkmuvafjdsjkadqftqsfnjdjlmdjxhjffjdpjnqkwjlmdfjknjpxgejwqsufmsfayjgfsqgxujdqkxgnqensknmmwsladfnjdcjhjwqjpjfnxfxgagyjdsfxgyqgemlmlljgsqpjfjkngqiajsqsjssjgfqxwlmdumagfqgexgjlljkfqpjyjljgsjxgyfnxffnjfmmwsxgykmglqeadxfqmglmkasjgkmagfjdjyqgyjljgsqpjkmuvjfqfqmgsymjsgmfwjxysfayjgfsfmogmcfnjqdjgjutxsjlljkfqpjwtxsfjxknqgefnjufmxkfqpjwtfnqgowqojxgxffxkojdvqkmkflqsxgmlljgsqpjwtmdqjgfjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgfnxfsjjosfmejgjdxfjqgfjdjsfqgkmuvafjdskqjgkjxumgenqensknmmwjdsfjxknqgefnjujgmaenxhmafkmuvafjdsjkadqftfmvqiajfnjqdkadqmsqftumfqpxfqgefnjufmjzvwmdjmgfnjqdmcgxgyjgxhwqgefnjufmhjffjdyjljgyfnjqduxknqgjsfnjlwxeqsvqkmKFL{G6D4U_4G41T515_15_73Y10A5_42JX1770}"

if my_model:
    mcmc_decrypt(target_cipher, my_model)

### Vigenère

**Reversed Encryption:**

Using the cipher word "CYLAB" we can reverse engineer the Vigenere Cipher to decrypt the flag.

In [5]:
# Vigenere Cipher Decryption
cipher_text = 'rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_f85729e7}'

keyword = 'CYLAB'

encryption_table = []
for i in range(26):
    encryption_table.append([chr((j + i) % 26 + ord('A')) for j in range(26)])

print('Encrypted Text: ', cipher_text)

plain_text = ''

key_offset = 0

for i in range(len(cipher_text)):
    if cipher_text[i].isalpha():
        row = ord(keyword[(i - key_offset) % len(keyword)]) - ord('A')
        col = ord(cipher_text[i].upper()) - ord('A')
        plain_char = chr((col - row) % 26 + ord('A'))

        if cipher_text[i].islower():
            plain_char = plain_char.lower()
        plain_text += plain_char
    
    else:
        plain_text += cipher_text[i]
        key_offset += 1


print('Plain Text: ', plain_text)

Encrypted Text:  rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_f85729e7}
Plain Text:  picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_d85729g7}


**Keyword Extraction by Pattern:**

Knowing that the flag starts with "picoCTF", we can try and match it to the start of the cipher text and try to find the cipher word.

In [26]:
def vigenere_decrypt(cipher_text, keyword):
    plain_text = ''
    key_offset = 0

    for i in range(len(cipher_text)):
        if cipher_text[i].isalpha():
            row = ord(keyword[(i - key_offset) % len(keyword)].upper()) - ord('A')
            col = ord(cipher_text[i].upper()) - ord('A')
            plain_char = chr((col - row) % 26 + ord('A'))

            if cipher_text[i].islower():
                plain_char = plain_char.lower()
            plain_text += plain_char
        
        else:
            plain_text += cipher_text[i]
            key_offset += 1

    return plain_text

def find_key_by_pattern_matching(ciphertext):
    print(f"Ciphertext: {ciphertext}")
    print("Assuming the plaintext starts with 'picoCTF{' pattern...")

    expected_start = "picoCTF"
    cipher_start = ""

    for char in ciphertext:
        if char.isalpha():
            cipher_start += char
        if len(cipher_start) >= len(expected_start):
            break

    print(f"Cipher start (alphabetic only): {cipher_start}")
    print(f"Expected start: {expected_start}")

    key = ""
    for i in range(len(expected_start)):
        cipher_char = cipher_start[i].upper()
        plain_char = expected_start[i].upper()

        cipher_pos = ord(cipher_char) - ord('A')
        plain_pos = ord(plain_char) - ord('A')

        key_pos = (cipher_pos - plain_pos) % 26
        key_char = chr(key_pos + ord('A'))
        key += key_char

    print(f"Derived key from pattern: {key}")

    for key_length in range(1, len(key) + 1):
        test_key = key[:key_length]
        decrypted = vigenere_decrypt(ciphertext, test_key)

        print(f"\nTesting key '{test_key}' (length {key_length}):")
        print(f"Decrypted: {decrypted}")

        if decrypted.startswith("picoCTF{") and decrypted.endswith("}"):
            print(f"Valid flag found with key: {test_key}")
            return decrypted, test_key

    return None, None

cipher_text = 'rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_f85729e7}'
decrypted_text, found_key = find_key_by_pattern_matching(cipher_text)
if decrypted_text:
    print(f"\nDecrypted flag: {decrypted_text}")
    print(f"Key used for decryption: {found_key}")

Ciphertext: rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_f85729e7}
Assuming the plaintext starts with 'picoCTF{' pattern...
Cipher start (alphabetic only): rgnoDVD
Expected start: picoCTF
Derived key from pattern: CYLABCY

Testing key 'C' (length 1):
Decrypted: pelmBTB{M0LS_UO3_E1E3M3R3_Y1YF3Q_d85729c7}

Testing key 'CY' (length 2):
Decrypted: pilqBXB{Q0LW_US3_E1I3M3V3_Y1CF3U_d85729g7}

Testing key 'CYL' (length 3):
Decrypted: picmFKB{Q0CS_YF3_E1I3D3R3_C1PF3U_u85729c7}

Testing key 'CYLA' (length 4):
Decrypted: picoBXS{O0LW_LQ3_E1I3D3T3_Y1CW3S_d85729g7}

Testing key 'CYLAB' (length 5):
Decrypted: picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_d85729g7}
Valid flag found with key: CYLAB

Decrypted flag: picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_d85729g7}
Key used for decryption: CYLAB


### picoCTF Flag Results

- Done - Guess My Cheese (Part 2): picoCTF{cHeEsY328d8e5d}
- Done - HideToSee: picoCTF{atbash_crack_05b2a65a}
- Done - Rotation: picoCTF{r0tat1on_d3crypt3d_7d140864}
- Done - substitution2: picoCTF{N6R4M_4N41Y515_15_73D10U5_42EA1770}
- Done - Vigenere: picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_d85729g7}